# 02 · พื้นฐาน Machine Learning (ช่วงที่ 2)

> **เป้าหมาย:** หลังจบ notebook นี้ เธอต้องบอกได้ว่า (1) train/validation/test ต่างกันอย่างไร (2) overfit เกิดยังไงและเห็นได้จากอะไร (3) Linear regression / Decision tree / Random forest ทำงานเชิงภาพรวมอย่างไร (4) ทำไมต้อง clean ข้อมูล ตัด outlier และทำ one-hot

**เวลาที่ใช้:** ~1.5 ชม. | ต้องใช้: numpy, pandas, matplotlib, scikit-learn

| ส่วน | หัวข้อ | ตรงกับโน้ตของเธอ |
|---|---|---|
| 1 | ML 3 ประเภท: supervised / unsupervised / reinforcement | `ML // supervised un reinforcement` |
| 2 | Train / Validation / Test | (พื้นฐานของ overfit) |
| 3 | Linear regression + MSE + gradient descent (ทำเอง) | `Linear regression`, `Metrics - mse` |
| 4 | **Overfit / underfit** (เห็นด้วยตา) | `overfit` |
| 5 | Decision tree → Random forest → Boosting (LightGBM/CatBoost) | `Random forest, Decision tree, Lightgbm, Catboost` |
| 6 | Clean value, outlier, one-hot encoding | `Clean value ตัดพวก outlier`, `One-hot encoding` |
| 7 | สรุป + แบบฝึก + resources | |

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(42)
print("numpy", np.__version__, "| pandas", pd.__version__)

## 1. ML 3 ประเภท

```
                    ┌────────────────────────────┐
                    │       Machine Learning     │
                    └──────────────┬─────────────┘
        ┌──────────────────────────┼───────────────────────────┐
  Supervised                 Unsupervised               Reinforcement
 (มีเฉลย/label)              (ไม่มีเฉลย)               (ลองผิดลองถูก + รางวัล)
 ข้อมูล = (x, y)             ข้อมูล = x อย่างเดียว         agent ↔ environment
 ทำนาย y จาก x               หาโครงสร้างในข้อมูล          เรียนรู้ policy ที่ได้รางวัลสูงสุด
 เช่น ภาพมือ→คำ              เช่น จัดกลุ่มท่ามือ            เช่น เกม, หุ่นยนต์
```

**โปรเจกต์ภาษามือของเธอเป็น supervised learning:** วิดีโอแต่ละคลิปมี label (คำที่ทำ) และเรา *ฝึกโมเดลให้ทำนาย label จากลำดับ landmarks*

Supervised แบ่งต่อเป็น **regression** (ทำนายตัวเลขต่อเนื่อง เช่น ราคาบ้าน) กับ **classification** (ทำนายหมวด เช่น คำที่ 17 จาก 20 คำ) ของเรา = classification

ลองดูตัวอย่างเล็ก ๆ ของแต่ละแบบ:

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression

X, y_true = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=3)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Supervised: มี label -> เรียนรู้ขอบเขตการตัดสินใจ
clf = LogisticRegression(max_iter=1000).fit(X, y_true)
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200), np.linspace(X[:,1].min()-1, X[:,1].max()+1, 200))
zz = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contourf(xx, yy, zz, alpha=0.25, cmap="viridis"); axes[0].scatter(X[:,0], X[:,1], c=y_true, cmap="viridis", edgecolor="k", s=25)
axes[0].set_title("SUPERVISED: labels given -> learn the boundary")

# Unsupervised: ไม่มี label -> ให้โมเดลหากลุ่มเอง
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
axes[1].scatter(X[:,0], X[:,1], c=km.labels_, cmap="Set1", edgecolor="k", s=25)
axes[1].scatter(*km.cluster_centers_.T, marker="X", s=200, c="black", label="cluster centers"); axes[1].legend()
axes[1].set_title("UNSUPERVISED (k-means): no labels -> find groups")
plt.tight_layout(); plt.show()

**Reinforcement learning (RL)** ต่างจากสองแบบข้างบน: ไม่มี "เฉลย" แต่มี *รางวัล* หลังลงมือทำ ตัวอย่างจิ๋ว: ทางเดิน 6 ช่อง เป้าหมายคือช่องขวาสุด agent ไม่รู้กฎ ต้องลองเดินซ้าย/ขวาแล้วเรียนรู้จากรางวัล (Q-learning)

In [ ]:
n_states, actions = 6, ["L", "R"]
Q = np.zeros((n_states, 2)); alpha, gamma, eps = 0.5, 0.9, 0.3
for episode in range(300):
    s = 0
    while s != n_states - 1:
        a = rng.integers(2) if rng.random() < eps else int(Q[s].argmax())   # สำรวจ vs ใช้สิ่งที่รู้
        s2 = max(0, s - 1) if a == 0 else min(n_states - 1, s + 1)
        r = 1.0 if s2 == n_states - 1 else 0.0                               # รางวัลเมื่อถึงเป้าหมายเท่านั้น
        Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])               # อัปเดตความเชื่อ
        s = s2
print("Q-table (แถว=ช่อง, คอลัมน์=[ซ้าย, ขวา]):"); print(Q.round(2))
print("policy ที่เรียนรู้ได้:", " ".join(actions[int(Q[s].argmax())] for s in range(n_states - 1)), "(R = เดินขวาไปหาเป้าหมาย)")

> 🧠 **ใจความ:** supervised = ครูมีเฉลย | unsupervised = ไม่มีเฉลย หาโครงสร้างเอง | RL = ไม่มีเฉลย แต่มีคะแนนหลังลงมือ
>
> 🧪 **Mini challenge 1.1:** เปลี่ยน `n_clusters` ของ k-means จาก 3 เป็น 5 แล้วดูผล โมเดลบังคับแบ่งเป็น 5 กลุ่มทั้งที่จริงมี 3 — บทเรียนคืออะไร? (unsupervised ไม่มีเฉลยมาบอกว่าถูกหรือผิด)

## 2. Train / Validation / Test

หัวใจของ ML: เป้าหมายคือโมเดลที่ทำนาย **ข้อมูลใหม่ที่ไม่เคยเห็น** ได้ดี ไม่ใช่แค่จำข้อมูลที่เคยเห็น ดังนั้นเราต้องกันข้อมูลบางส่วนไว้ทดสอบ

```
ข้อมูลทั้งหมด ──┬── TRAIN (~70%)  → ใช้ฝึก: โมเดลปรับน้ำหนักจากตรงนี้
               ├── VALIDATION (~15%) → ใช้ "ตัดสินใจระหว่างทาง": เลือกโมเดล, ปรับ hyperparameter, ดู overfit
               └── TEST (~15%)    → ใช้ "สอบจริง" ครั้งเดียวตอนจบ ห้ามแอบดูตอนปรับโมเดล
```

เปรียบเทียบ: train = แบบฝึกหัด, validation = ข้อสอบจำลอง, test = ข้อสอบจริง ถ้าเอาข้อสอบจริงมาใช้ฝึก คะแนนจะสวยแต่ไม่มีความหมาย

ข้อควรระวังของโปรเจกต์นี้ (จาก notebook 01): แบ่ง **ตามผู้แสดง** ไม่ใช่สุ่มระดับคลิป

In [ ]:
from sklearn.model_selection import train_test_split

X = np.arange(100).reshape(-1, 1); y = (X[:, 0] % 2)       # ข้อมูลตัวอย่าง 100 แถว
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.15/0.85, random_state=0)
print("train:", len(X_train), "| val:", len(X_val), "| test:", len(X_test))

fig, ax = plt.subplots(figsize=(9, 1.4))
colors = np.full(100, "#60a5fa", dtype=object); colors[X_val[:,0]] = "#fbbf24"; colors[X_test[:,0]] = "#f87171"
ax.scatter(range(100), np.zeros(100), c=list(colors), s=60, marker="s"); ax.axis("off")
ax.set_title("blue=train, yellow=validation, red=test (สุ่มกระจาย)", fontsize=10); plt.show()

## 3. Linear regression + MSE + Gradient descent (ทำเองทีละขั้น)

โมเดลที่ง่ายที่สุด: **เส้นตรง** `ŷ = w·x + b` (w = ความชัน, b = จุดตัดแกน) เป้าหมายคือหา `w, b` ที่ทำให้เส้นผ่านจุดข้อมูลได้ดีที่สุด

**วัด "ดี" ด้วย loss:** Mean Squared Error  `MSE = (1/N) Σ (ŷᵢ - yᵢ)²`  
ยกกำลังสองเพราะ (1) ไม่ให้บวกลบหักล้างกัน (2) ลงโทษความผิดพลาดใหญ่หนักกว่า

**หาค่าที่ดีที่สุดด้วย gradient descent:** จินตนาการ MSE เป็นภูเขา (แกน = w, b) เราอยู่บนเนินแล้วก้าวลงไปทางที่ชันลงที่สุดทีละก้าว

```
w ← w − lr · ∂MSE/∂w         lr = learning rate (ขนาดก้าว)
b ← b − lr · ∂MSE/∂b
```

In [ ]:
# ข้อมูลจริงมาจาก y = 3x + 2 + noise
x = rng.uniform(0, 1, 50); y = 3 * x + 2 + rng.normal(0, 0.3, 50)

def mse(w, b): return np.mean((w * x + b - y) ** 2)

def gradient_descent(lr, steps=60, w0=-2.0, b0=-1.0):
    w, b = w0, b0; path = [(w, b, mse(w, b))]
    for _ in range(steps):
        err = w * x + b - y
        dw, db = 2 * np.mean(err * x), 2 * np.mean(err)     # อนุพันธ์ของ MSE
        w, b = w - lr * dw, b - lr * db
        path.append((w, b, mse(w, b)))
    return np.array(path)

path = gradient_descent(lr=0.3)
print("เริ่มที่ w=%.2f b=%.2f MSE=%.2f" % tuple(path[0]))
print("จบที่   w=%.2f b=%.2f MSE=%.3f  (สมการที่ใช้สร้างข้อมูลคือ w=3, b=2 แต่มี noise)" % tuple(path[-1]))

# เปรียบเทียบกับคำตอบสำเร็จรูป
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression().fit(x.reshape(-1, 1), y); print("sklearn (คำตอบที่ดีที่สุดของข้อมูลชุดนี้): w=%.2f b=%.2f" % (lr_model.coef_[0], lr_model.intercept_))

In [ ]:
ws, bs = np.linspace(-3, 6, 120), np.linspace(-2, 5, 120)
Z = np.array([[mse(w, b) for w in ws] for b in bs])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
axes[0].contourf(ws, bs, np.log(Z), levels=25, cmap="viridis"); axes[0].plot(path[:,0], path[:,1], "w.-", lw=1)
axes[0].scatter(*path[0,:2], c="red", s=50, label="start"); axes[0].scatter(*path[-1,:2], c="yellow", s=60, label="end")
axes[0].set_xlabel("w"); axes[0].set_ylabel("b"); axes[0].set_title("Loss landscape (log MSE) + path"); axes[0].legend()

axes[1].scatter(x, y, s=18, alpha=0.7)
xs = np.linspace(0, 1, 2)
for k, c in zip([0, 3, 10, 60], ["#ef4444", "#f59e0b", "#10b981", "#2563eb"]):
    axes[1].plot(xs, path[k,0] * xs + path[k,1], color=c, label=f"step {k}")
axes[1].set_title("Line improves with each step"); axes[1].legend()

axes[2].plot(path[:,2]); axes[2].set_yscale("log"); axes[2].set_xlabel("step"); axes[2].set_ylabel("MSE"); axes[2].set_title("LOSS CURVE"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**สังเกต:** (0) หลัง 60 ก้าว GD ได้ w ≈ 2.4 ซึ่งยัง *ไม่ถึง* คำตอบที่ดีที่สุดของ sklearn (w ≈ 2.75) นิดหน่อย — ก้นหลุมแคบและแบน เลยลงช้าช่วงท้าย ลองเพิ่ม `steps` เป็น 500 แล้วดูว่าเข้าใกล้ขึ้นไหม (ค่า w ที่ได้ไม่ใช่ 3 เป๊ะ เพราะ noise ในข้อมูล) (1) เส้นสีแดง→น้ำเงินขยับเข้าหาข้อมูล (2) จุดสีขาวไต่ลงหุบเขาเข้าหาก้นหลุม (3) **loss curve** ลดลงเร็วตอนแรกแล้วค่อยๆ แบน — นี่คือรูปทรงปกติที่ต้องคุ้นตา (จะเจออีกใน notebook 04)

### ผลของ learning rate

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for lr, c in [(0.02, "#9ca3af"), (0.3, "#2563eb"), (0.75, "#f59e0b"), (0.95, "#dc2626")]:
    p = gradient_descent(lr, steps=40); ax.plot(np.clip(p[:,2], 1e-3, 1e3), color=c, label=f"lr={lr}")
ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("MSE (clipped)"); ax.legend(); ax.grid(alpha=0.3); ax.set_title("Effect of learning rate")
plt.show()
for lr in [0.02, 0.3, 0.75, 0.95]:
    print(f"lr={lr:<5} final MSE = {gradient_descent(lr, steps=40)[-1,2]:.4g}")

- **lr น้อยเกิน** (เทา): ก้าวจิ๋ว ลงช้ามาก ในเวลาจำกัดยังไปไม่ถึงก้นหลุม
- **lr พอดี** (น้ำเงิน): ลงเร็วและนิ่ง
- **lr 0.75** (ส้ม): ใหญ่กว่าเส้นน้ำเงิน แต่ยังไม่เกินขอบ — ช่วงแรกช้ากว่า (เส้นโค้งลงเนียน) แต่ท้ายสุดลงได้ต่ำสุดของทั้งสี่เส้น บทเรียน: lr ที่ใหญ่ขึ้นช่วยได้ *จนถึงขอบ* แล้วพังทันที (ขอบประมาณ 0.8 สำหรับข้อมูลนี้)
- **lr 0.95** (แดง): ก้าวข้ามหุบเขาไปไกลกว่าเดิมทุกครั้ง **diverge** (กราฟตัดแกนที่ 1000 เส้นจึงแบน แต่ตัวเลขที่ print ออกมาจริงๆ ใหญ่ระดับ 10¹²)

> 🧪 **Mini challenge 3.1:** ลองไล่ค่า `lr` ระหว่าง 0.7 ถึง 0.9 เพื่อหาขอบที่เริ่ม diverge (สำหรับข้อมูลชุดนี้ประมาณ 0.8) ทำไมจึงมี "ขอบ" แบบนั้น? (คำใบ้: ถ้าก้าวใหญ่กว่าความกว้างของหุบเขา จะกระโดดไปฝั่งตรงข้ามที่สูงกว่าเดิม)
>
> 🧪 **Mini challenge 3.2:** เปลี่ยนจุดเริ่ม `w0, b0` ใน `gradient_descent` ไปไกลๆ ก้นหลุมของ linear regression มีจุดเดียว จึงลงจุดไหนก็ถึงที่เดิม (ต่างจากโครงข่ายประสาทที่จะเจอใน notebook 04)

## 4. Overfit / Underfit — เห็นด้วยตา

ทดลองฟิตเส้นโค้งพหุนามหลายระดับ (degree) กับข้อมูลที่จริงๆ คือคลื่น sin + noise **ใช้ข้อมูลฝึกน้อยจุดโดยตั้งใจ**

- degree ต่ำเกิน → โมเดลไม่ยืดหยุ่นพอ = **underfit** (high bias)
- degree สูงเกิน → โมเดลจำ noise ของข้อมูลฝึก = **overfit** (high variance)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def true_f(x): return np.sin(2 * np.pi * x)
x_tr = rng.uniform(0, 1, 20);  y_tr = true_f(x_tr) + rng.normal(0, 0.25, 20)
x_va = rng.uniform(0, 1, 200); y_va = true_f(x_va) + rng.normal(0, 0.25, 200)
grid = np.linspace(0, 1, 300)

def fit_poly(d):
    return make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression()).fit(x_tr.reshape(-1,1), y_tr)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, d, name in zip(axes, [1, 4, 12], ["degree 1: UNDERFIT", "degree 4: GOOD FIT", "degree 12: OVERFIT"]):
    m = fit_poly(d)
    tr_e = mean_squared_error(y_tr, m.predict(x_tr.reshape(-1,1))); va_e = mean_squared_error(y_va, m.predict(x_va.reshape(-1,1)))
    ax.scatter(x_tr, y_tr, c="k", s=25, label="train points"); ax.plot(grid, true_f(grid), "g--", label="true function")
    ax.plot(grid, np.clip(m.predict(grid.reshape(-1,1)), -2.5, 2.5), "r", label="model"); ax.set_ylim(-2.5, 2.5)
    ax.set_title(f"{name}\ntrain MSE={tr_e:.3f} | val MSE={va_e:.3f}", fontsize=10); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
degrees = range(1, 15)
tr_err, va_err = [], []
for d in degrees:
    m = fit_poly(d)
    tr_err.append(mean_squared_error(y_tr, m.predict(x_tr.reshape(-1,1)))); va_err.append(mean_squared_error(y_va, m.predict(x_va.reshape(-1,1))))

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(degrees, tr_err, "o-", label="train error"); ax.plot(degrees, va_err, "s-", label="validation error")
ax.set_yscale("log"); ax.set_xlabel("model complexity (polynomial degree)"); ax.set_ylabel("MSE"); ax.grid(alpha=0.3)
best = degrees[int(np.argmin(va_err))]; ax.axvline(best, color="gray", ls=":"); ax.text(best + 0.2, ax.get_ylim()[1] * 0.3, f"best val\ndegree={best}")
ax.legend(); ax.set_title("THE overfitting picture"); plt.show()

### นี่คือภาพที่สำคัญที่สุดของ notebook นี้
- **train error ลดลงเรื่อยๆ** เมื่อโมเดลซับซ้อนขึ้น (โมเดลยืดหยุ่นขึ้นย่อมจำข้อมูลฝึกได้ดีขึ้น)
- **validation error ลดแล้วกลับ *เพิ่มขึ้น*** → ตรงที่มันเริ่มแยกจากเส้น train คือจุดเริ่ม overfit
- ถ้าดูแค่ train error จะหลงว่าโมเดลซับซ้อนที่สุดดีที่สุด — **ต้องดู validation เสมอ**

**วิธีอ่านสัญญาณ (จะใช้ซ้ำใน notebook 04 กับ loss curve ของ deep learning):**
| อาการ | ความหมาย |
|---|---|
| train สูง, val สูง | underfit — โมเดลเล็ก/ฝึกไม่พอ |
| train ต่ำ, val ต่ำ ใกล้กัน | ดี |
| train ต่ำมาก, val สูงกว่ามาก | **overfit** |

**วิธีแก้ overfit:** ข้อมูลมากขึ้น, โมเดลง่ายขึ้น, regularization (ลงโทษน้ำหนักใหญ่), early stopping, data augmentation, dropout (สองอย่างหลังสำหรับ deep learning)

> 🧪 **Mini challenge 4.1:** เปลี่ยนข้อมูลฝึกจาก 20 จุดเป็น 200 จุด (`rng.uniform(0,1,200)`) แล้วรันกราฟ "overfitting picture" ใหม่ degree 12 ยัง overfit ไหม? (บทเรียน: ข้อมูลมากขึ้นลดการ overfit)
>
> 🧪 **Mini challenge 4.2:** แทน `LinearRegression()` ด้วย `Ridge(alpha=1.0)` (นำเข้าจาก `sklearn.linear_model`) แล้วดูว่า degree 12 ดีขึ้นไหม นี่คือ regularization
>
> ❓ **คิดดู:** ถ้าเราปรับ degree โดยดู *test* error หลายรอบ แล้วเลือกตัวที่ test ต่ำสุด เกิดปัญหาอะไร? (คำใบ้: test ถูก "ใช้ฝึก" ทางอ้อม)

## 5. Decision Tree → Random Forest → Boosting

### Decision tree: ต้นไม้คำถามใช่/ไม่ใช่
ต้นไม้แบ่งข้อมูลด้วยคำถามเช่น "feature ที่ 2 > 0.5?" ทีละชั้น จนกลุ่มย่อยปลายใบ "บริสุทธิ์" (เป็นคลาสเดียว)

**ข้อดี:** อ่านง่าย ไม่ต้อง scale ข้อมูล  **ข้อเสีย:** ลึกไปก็จำ noise (overfit ง่ายมาก)

ดูผลของความลึก (`max_depth`) บนข้อมูล 2 มิติ:

In [ ]:
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

Xm, ym = make_moons(n_samples=400, noise=0.35, random_state=1)
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.4, random_state=1)

def plot_boundary(ax, model, title):
    xx, yy = np.meshgrid(np.linspace(-2, 3, 250), np.linspace(-1.5, 2, 250))
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm"); ax.scatter(Xtr[:,0], Xtr[:,1], c=ytr, cmap="coolwarm", edgecolor="k", s=18)
    ax.set_title(f"{title}\ntrain acc={model.score(Xtr, ytr):.2f} | test acc={model.score(Xte, yte):.2f}", fontsize=10)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, d in zip(axes, [1, 4, None]):
    plot_boundary(ax, DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr), f"Decision tree (max_depth={d})")
plt.tight_layout(); plt.show()

small = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xtr, ytr)
print(export_text(small, feature_names=["x1", "x2"]))

**สังเกต:** depth 1 ตัดเส้นเดียว = underfit | depth 4 พอดี | ไม่จำกัด = ขอบเขตหยักแหลมล้อมทุกจุด **train acc = 1.00 แต่ test acc (~0.78) ไม่ได้ดีไปกว่า depth 4 (~0.81)** ช่องว่างระหว่าง train กับ test กว้างที่สุด (overfit แบบเดียวกับหัวข้อ 4 แต่เห็นเป็นรูปร่างขอบเขต) หมายเหตุ: test มีแค่ 160 จุด ความต่าง 2-3% อยู่ในระดับที่ผันผวนตามการสุ่มได้ ให้ดู *แนวโน้ม* มากกว่าตัวเลขเป๊ะ

### Random Forest: ต้นไม้หลายต้นโหวต
เทคนิค **bagging**: สร้างต้นไม้หลายต้น แต่ละต้นเห็นข้อมูลที่สุ่มมา (และ feature สุ่ม) แล้ว **โหวตเฉลี่ย** ต้นเดี่ยวๆ อาจ overfit แต่ข้อผิดพลาดที่ต่างกันไปมักหักล้างกันเมื่อเฉลี่ย

ในรูปด้านล่าง ฟอเรสต์ได้ test acc ใกล้เคียงหรือดีกว่าต้นเดี่ยวเล็กน้อย บนข้อมูลเล็กความต่างอาจไม่ชัดมาก ลองทำ challenge 5.2 เพื่อเห็นผลชัดขึ้น

### Boosting (Gradient Boosting / LightGBM / CatBoost): ต้นไม้ต่อกันแก้ความผิดพลาด
สร้างต้นไม้เล็กๆ ทีละต้น โดยต้นถัดไป **เน้นแก้ส่วนที่ต้นก่อนหน้าทำนายพลาด** (ต่างจาก forest ที่สร้างขนานกันอิสระ) LightGBM และ CatBoost คือ *การ implement ที่เร็วและแม่นยำ* ของแนวคิดนี้ (CatBoost จัดการ categorical feature ได้ดีเป็นพิเศษ)

In [ ]:
models = {
    "Single tree (no limit)":    DecisionTreeClassifier(random_state=0),
    "Random forest (100 trees)": RandomForestClassifier(n_estimators=100, random_state=0),
    "Gradient boosting":         GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.1, random_state=0),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (name, m) in zip(axes, models.items()):
    plot_boundary(ax, m.fit(Xtr, ytr), name)
plt.tight_layout(); plt.show()

> ℹ️ **เรื่อง LightGBM / CatBoost:** สองไลบรารีนี้ไม่ได้ติดตั้งในสภาพแวดล้อมที่ใช้สร้าง notebook นี้ (ติดตั้งแพ็กเกจไม่ได้) จึง **ไม่ได้ทดสอบโค้ดของสองตัวนี้** ใน notebook นี้ ใช้ `GradientBoostingClassifier` ของ sklearn แทนเพื่ออธิบายแนวคิดเดียวกัน ถ้าอยากลองบนเครื่องตัวเอง (`pip install lightgbm catboost`) รูปแบบการใช้งานคือ:
> ```python
> from lightgbm import LGBMClassifier;  m = LGBMClassifier(n_estimators=100).fit(Xtr, ytr)
> from catboost import CatBoostClassifier; m = CatBoostClassifier(iterations=100, verbose=0).fit(Xtr, ytr)
> ```
> สำหรับโปรเจกต์นี้ ต้นไม้เหล่านี้ใช้เป็น **baseline เทียบ** (เช่นเอา feature สรุปจากลำดับ landmarks ไปฝึก) ไม่ใช่โมเดลหลัก

> 🧪 **Mini challenge 5.1:** ปรับ `n_estimators` ของ random forest เป็น 1, 5, 100 กราฟขอบเขตและ test acc เปลี่ยนอย่างไร?
>
> 🧪 **Mini challenge 5.2:** ปรับ `noise` ของ `make_moons` เป็น 0.1 แล้ว 0.6 ต้นไม้ลึกไม่จำกัดยังแย่กว่าฟอเรสต์ไหม? ทำไม?
>
> ❓ **คิดดู:** ทำไมต้นไม้ไม่ต้อง scale ข้อมูล (ต่างจาก KNN/โครงข่ายประสาท)? (คำใบ้: ต้นไม้ถามแค่ "มากกว่า/น้อยกว่าค่าหนึ่ง")

## 6. Clean value, Outlier, One-hot encoding

ข้อมูลจริงสกปรกเสมอ: ค่าหาย, ค่าผิดปกติ, หมวดหมู่เป็นข้อความ โมเดลส่วนใหญ่รับแต่ตัวเลข จึงต้องเตรียมก่อน

### 6.1 Outlier: ค่าที่หลุดจากพวก
เพราะ MSE ยกกำลังสอง **outlier เพียงไม่กี่จุดดึงเส้นให้เพี้ยนได้มาก**

In [ ]:
years = rng.uniform(0, 10, 40); salary = 20 + 5 * years + rng.normal(0, 4, 40)       # เงินเดือน (พันบาท)
years_o = np.r_[years, [1.0, 2.0, 1.5]]; salary_o = np.r_[salary, [120, 130, 110]]  # เพิ่ม outlier 3 จุด (พิมพ์ผิด/ข้อมูลเพี้ยน)

def iqr_mask(v, k=1.5):
    q1, q3 = np.percentile(v, [25, 75]); iqr = q3 - q1
    return (v >= q1 - k * iqr) & (v <= q3 + k * iqr)

# ตรวจ outlier จากส่วนเหลือ (residual) ของเส้นแรกเริ่ม: ใช้ IQR บน salary เทียบกับค่าคาดการณ์
fit0 = np.polyfit(years_o, salary_o, 1); resid = salary_o - np.polyval(fit0, years_o)
keep = iqr_mask(resid)
print("จุดทั้งหมด:", len(salary_o), "| ถูกตัดเป็น outlier:", (~keep).sum())

fit_all, fit_clean = np.polyfit(years_o, salary_o, 1), np.polyfit(years_o[keep], salary_o[keep], 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(years_o[keep], salary_o[keep], s=22, label="kept"); axes[0].scatter(years_o[~keep], salary_o[~keep], c="r", marker="x", s=70, label="flagged outlier")
xs = np.array([0, 10]); axes[0].plot(xs, np.polyval(fit_all, xs), "r--", label=f"fit with outliers (slope={fit_all[0]:.1f})")
axes[0].plot(xs, np.polyval(fit_clean, xs), "g-", label=f"fit after cleaning (slope={fit_clean[0]:.1f})"); axes[0].plot(xs, 20 + 5 * xs, "k:", label="truth (slope=5)")
axes[0].legend(fontsize=8); axes[0].set_xlabel("years"); axes[0].set_ylabel("salary"); axes[0].set_title("Outliers drag the line")
axes[1].boxplot(resid, vert=False); axes[1].set_title("Boxplot of residuals: points beyond whiskers = outliers"); plt.tight_layout(); plt.show()

**อ่านผล:** เส้นสีแดงเอียงผิดไปเพราะ 3 จุดเพี้ยน หลังตัดออกเส้นเขียวใกล้ความจริงมาก

**กฎ IQR:** ค่าที่ต่ำกว่า `Q1 − 1.5·IQR` หรือสูงกว่า `Q3 + 1.5·IQR` ถือเป็น outlier (นี่คือที่มาของ "หนวด" ใน boxplot)

⚠️ **ระวัง:** ไม่ใช่ outlier ทุกตัวต้องตัด บางตัวคือ *ข้อมูลจริงที่หายาก* (เช่นคนมือใหญ่ผิดปกติ) ตัดผิดอาจเสียข้อมูลสำคัญ ให้ถามก่อนว่า "ค่านี้เป็นความผิดพลาดหรือเป็นของจริง?" ในโปรเจกต์ภาษามือ outlier ที่พบบ่อยคือ *landmark กระโดดผิดตำแหน่งชั่วเฟรม* (detector พลาด) ซึ่งควรจัดการ

### 6.2 ค่าหาย (missing values) และ scaling

In [ ]:
df = pd.DataFrame({"age": [25, 31, np.nan, 40, 29], "height_cm": [170, np.nan, 165, 180, 175], "city": ["Bangkok", "Chiang Mai", "Bangkok", "Phuket", None]})
print("ก่อนทำความสะอาด:"); print(df); print("\nจำนวนค่าหายต่อคอลัมน์:"); print(df.isna().sum())

clean = df.copy()
clean["age"] = clean["age"].fillna(clean["age"].median())              # ตัวเลข: เติมด้วย median (ทนต่อ outlier กว่า mean)
clean["height_cm"] = clean["height_cm"].fillna(clean["height_cm"].median())
clean["city"] = clean["city"].fillna("Unknown")                        # ข้อความ: เติมด้วยหมวด "Unknown"
print("\nหลังทำความสะอาด:"); print(clean)

# scaling: ปรับให้ทุกคอลัมน์มี mean 0, std 1 (สำคัญต่อ KNN / โครงข่ายประสาท)
from sklearn.preprocessing import StandardScaler
scaled = StandardScaler().fit_transform(clean[["age", "height_cm"]]); print("\nหลัง StandardScaler: mean =", scaled.mean(0).round(6), "std =", scaled.std(0).round(3))

### 6.3 One-hot encoding
โมเดลคำนวณกับตัวเลข ถ้าเราแปลงเมืองเป็น `Bangkok=0, Chiang Mai=1, Phuket=2` โมเดลจะ "เข้าใจผิด" ว่า Phuket > Chiang Mai > Bangkok และเฉลี่ย Bangkok กับ Phuket ได้ Chiang Mai ซึ่งไร้สาระ **One-hot** แก้โดยให้แต่ละหมวดมีคอลัมน์ของตัวเอง (0/1):

```
city        →   city_Bangkok  city_Chiang Mai  city_Phuket  city_Unknown
Bangkok     →        1              0               0            0
Chiang Mai  →        0              1               0            0
```

In [ ]:
print("--- pandas.get_dummies ---")
print(pd.get_dummies(clean["city"], prefix="city", dtype=int))

from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(clean[["city"]])
print("\n--- sklearn OneHotEncoder (ใช้ใน pipeline จริงได้ จำค่าจาก train แล้วใช้กับ test) ---")
print("categories ที่เรียนรู้:", enc.categories_[0])
print("เมืองใหม่ที่ไม่เคยเห็น ('Khon Kaen') ->", enc.transform(pd.DataFrame({"city": ["Khon Kaen"]})))

**สังเกต:** `OneHotEncoder` ของ sklearn *จำ* หมวดที่เห็นตอนฝึก และจัดการหมวดใหม่ (`handle_unknown="ignore"` ให้ศูนย์ทั้งแถว) ตอนใช้จริง ซึ่งปลอดภัยกว่า `get_dummies` ที่คอลัมน์ขึ้นกับข้อมูลที่ส่งเข้ามา

**เชื่อมกับโปรเจกต์:** feature ของเรา (พิกัด landmarks) เป็นตัวเลขอยู่แล้วจึงไม่ต้อง one-hot แต่ **label** (ชื่อคำ) ต้องแปลงเป็นเลขคลาส (0,1,2,…) เสมอ — ตรงนี้ label encoding ใช้ได้เพราะโมเดลมองเป็นคลาส ไม่ใช่ขนาด

> 🧪 **Mini challenge 6.1:** เพิ่มคอลัมน์ `"size": ["S","M","L","M","S"]` แล้วคิดว่าควร one-hot หรือแปลงเป็น 0,1,2? (คำใบ้: ขนาดมี *ลำดับ* จริง)
>
> 🧪 **Mini challenge 6.2:** ลอง `k=3.0` ใน `iqr_mask` แล้วดูว่าตัด outlier เหลือกี่จุด ค่า k ใหญ่ = ผ่อนปรน

## 7. สรุป + Resources

**ที่ควรอธิบายได้หลังจบ notebook นี้**
1. supervised / unsupervised / RL ต่างกันที่ "มีเฉลยไหม" และเราทำ supervised classification
2. train ฝึก, validation ตัดสินใจ, test สอบจริงครั้งเดียว
3. MSE = กำลังสองของความผิดพลาดเฉลี่ย; gradient descent = ก้าวลงเขา loss ทีละก้าว; learning rate = ขนาดก้าว
4. overfit = train ดีมาก แต่ validation แย่ลง (ดูได้จากกราฟสองเส้นแยกจากกัน)
5. ต้นไม้ → ฟอเรสต์ (เฉลี่ยหลายต้น) → boosting (ต่อกันแก้ผิด)
6. clean: จัดการค่าหาย, outlier (ระวังตัดของจริง), one-hot สำหรับ feature หมวดหมู่

### ควรอ่าน/ดูเพิ่ม (ค้นยืนยันแล้วว่ามีอยู่จริง)
| หัวข้อ | แหล่ง |
|---|---|
| Overfit / generalization / loss curve | Google ML Crash Course — module *Datasets, generalization, and overfitting*: https://developers.google.com/machine-learning/crash-course/overfitting/overfitting |
| Linear regression + gradient descent + learning rate | Google ML Crash Course: https://developers.google.com/machine-learning/crash-course |
| Bias vs Variance, Cross validation, Decision Trees, Random Forests | วิดีโอ **StatQuest** (Josh Starmer) — playlist "Machine Learning": https://www.classcentral.com/course/youtube-machine-learning-90293 (รายการวิดีโอ ลิงก์ไป YouTube) |
| sklearn + pandas + matplotlib แบบ notebook | Python Data Science Handbook: https://github.com/jakevdp/PythonDataScienceHandbook (อ่านออนไลน์: https://jakevdp.github.io/PythonDataScienceHandbook/) |

> ⏭️ **ต่อไป:** `03_metrics_sklearn.ipynb` — วัดผลให้ถูก: accuracy ไม่พอ, precision/recall, macro-F1, confusion matrix